# Sesión 11 — Afinar sin hacer trampa: gradient boosting, Optuna y el modelo final

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 4 — Evaluación y optimización · **se presenta E4**

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Explicar cómo funciona el **gradient boosting** (árboles en fila que corrigen residuales) y entrenarlo con parada temprana.
2. Buscar hiperparámetros con `GridSearchCV`, `RandomizedSearchCV` y **Optuna**, y explicar por qué la búsqueda también es optimización.
3. Reconocer que el mejor puntaje de una búsqueda es **optimista** y reportar el desempeño final de forma honesta.
4. Explicar una predicción con **SHAP** y guardar el `Pipeline` final con `joblib` para usarlo con datos nuevos.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
                                       ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲
   HOY: el modelo final — elegirlo, afinarlo, medirlo honestamente, explicarlo y guardarlo.
```

🔍 **Predice** · ✏️ **Completa** (andamiaje **bajo**) · 💥 **Trampa** · 🖼️ **Fichas** · 📦 **Entrega E4**.

> **Datos:** Bank Marketing (S5, S6, S10). **Herramientas nuevas:** `HistGradientBoostingClassifier` (viene en scikit-learn), **Optuna** (se instala en la primera celda) y **SHAP** (ya viene en Colab).


## 0. Setup

In [ ]:
%pip install -q optuna

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 40)
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVOS = {"bank.csv": "datos/bank.csv", "figuras.py": "tools/figuras.py"}

def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo}. Use el Plan B (celda siguiente).")
    return False

for archivo, ruta in ARCHIVOS.items():
    asegurar_datos(archivo, REPO + ruta)
import figuras as fg
print("Listo:", ", ".join(ARCHIVOS))

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar algún archivo</summary>

La celda siguiente revisa sola qué falta y solo en ese caso le pide subirlo.
</details>

In [ ]:
# Plan B: solo actúa si falta algún archivo. Con "Ejecutar todo" no se detiene.
from pathlib import Path
faltan = [a for a in ARCHIVOS if not Path(a).exists()]
if not faltan:
    print("✓ Todo disponible: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ Faltan:", faltan, "→ súbalos")
    files.upload()

In [ ]:
fg.pipeline(["modelo", "evaluación", "interpretación"], modulo=4, subtitulo="HOY: el modelo final — afinarlo, medirlo honestamente, explicarlo y guardarlo");

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

bank = pd.read_csv("bank.csv")
y = (bank["deposit"] == "yes").astype(int)
X = pd.get_dummies(bank.drop(columns=["deposit", "duration"]), drop_first=True, dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(X_train.shape, X_test.shape)

### 🔍 Predice
En la Sesión 10 el bosque llegó a AUC **0,792** en validación cruzada. Con el modelo más usado en la industria y una búsqueda automática de hiperparámetros, ¿a cuánto cree que llegamos hoy?

**Tu predicción:** AUC ≈ ____

## 1. Gradient boosting: árboles en fila

El bosque (Sesión 6) entrena cientos de árboles **en paralelo**, cada uno por su lado, y los promedia. El **gradient boosting** los entrena **en fila**: cada árbol nuevo, pequeño, se dedica a corregir lo que los anteriores todavía no explican (los **residuales**). Se ve mejor con una sola variable:

In [ ]:
fg.boosting_paso_a_paso(etapas=(1, 3, 10, 100));

### 📐 Fundamento
Después de $m$ árboles, la predicción es la suma de todos, cada uno con un peso pequeño $\eta$:

$$F_m(x) = F_{m-1}(x) + \eta \, h_m(x)$$

- $h_m$ es un árbol pequeño entrenado para predecir los residuales de $F_{m-1}$ (en general, el **gradiente** de la pérdida: por eso se llama *gradient* boosting).
- $\eta$ es el `learning_rate`, **el mismo de la Sesión 4**: pasos pequeños → hacen falta más árboles, pero se sobreajusta menos.
- En clasificación se suman *log-odds* y al final se pasa por la sigmoide, igual que la logística (Sesión 5).

<details><summary><b>Para quien quiera más:</b> HistGradientBoosting, LightGBM, XGBoost, CatBoost</summary>

Son implementaciones del mismo algoritmo con trucos de velocidad. `HistGradientBoostingClassifier` (scikit-learn) y LightGBM agrupan cada variable numérica en ~255 intervalos (histogramas), lo que los hace muy rápidos con muchas filas; XGBoost fue el que popularizó el método en Kaggle; CatBoost maneja muy bien las variables categóricas. Para este curso, `HistGradientBoostingClassifier` basta y no hay que instalar nada.
</details>

![Dentro del .fit() — gradient boosting](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S11/img/dentro_del_fit_boosting.png)

### ✏️ Completa — boosting con valores por defecto
Haga la validación cruzada de un `HistGradientBoostingClassifier(random_state=42)` con el mismo `cv` de siempre y `scoring="roc_auc"`. Fíjese en el tiempo: el bosque tardaba ~15 s.

In [ ]:
import time
from sklearn.ensemble import HistGradientBoostingClassifier

t = time.time()
scores_hgb = ...   # TODO: cross_val_score del HistGradientBoostingClassifier(random_state=42)
print(f"boosting por defecto: {scores_hgb.mean():.3f} ± {scores_hgb.std():.3f}   ({time.time() - t:.1f} s)")

**0,796** en unos segundos: ya le gana un poquito al bosque sin tocar nada. ¿Cuántos árboles usó? Lo decide solo, con **parada temprana**: aparta un 10 % del entrenamiento y deja de agregar árboles cuando esa validación interna deja de mejorar.

In [ ]:
hgb = HistGradientBoostingClassifier(early_stopping=True, max_iter=500, random_state=42).fit(X_train, y_train)
fg.boosting_curva(hgb);

## 2. Hiperparámetros: las perillas que el `.fit()` no aprende

Los **parámetros** los aprende el `.fit()` (los cortes de cada árbol, los coeficientes de la logística). Los **hiperparámetros** los decide usted antes: `learning_rate`, `max_leaf_nodes`, `k` en k-NN, `C` en la logística. Elegirlos es otra optimización: maximizar el AUC de **validación cruzada**, sin gradiente, donde cada evaluación cuesta 5 entrenamientos.

![Cómo buscar hiperparámetros](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S11/img/como_buscar_hiperparametros.png)

### La grilla
`GridSearchCV` prueba todas las combinaciones de una lista. Aquí 3 × 3 = 9 combinaciones × 5 folds = 45 entrenamientos.

### ✏️ Completa — la grilla
Defina `grilla` con `learning_rate` en `[0.03, 0.1, 0.3]` y `max_leaf_nodes` en `[7, 15, 31]`.

In [ ]:
from sklearn.model_selection import GridSearchCV

grilla = ...   # TODO: diccionario {"nombre_del_hiperparámetro": [valores, ...], ...}
busqueda = GridSearchCV(HistGradientBoostingClassifier(random_state=42), grilla, cv=cv, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
print("mejor combinación:", busqueda.best_params_, f"· AUC {busqueda.best_score_:.3f}")
fg.grid_heatmap(busqueda.cv_results_, "learning_rate", "max_leaf_nodes");

**Lectura:** la mejor casilla (0,797) le gana a la configuración por defecto (0,796) por **0,001**, y seis de las nueve casillas están a menos de 0,005. Los valores por defecto de scikit-learn ya son buenos; la grilla casi no encontró nada.

### ¿Y la búsqueda al azar?
Con el mismo presupuesto, `RandomizedSearchCV` prueba combinaciones al azar. Parece peor, pero suele ganar (Bergstra y Bengio, 2012):

In [ ]:
fg.grid_vs_random();

## 3. Optuna: una búsqueda que aprende

**Optuna** es hoy la herramienta estándar para buscar hiperparámetros. En vez de probar a ciegas, usa los intentos anteriores para proponer el siguiente (un método bayesiano llamado *TPE*): explora al principio y después se concentra donde le ha ido bien.

Se usa en tres pasos:
1. Una función `objetivo(trial)` que **propone** hiperparámetros con `trial.suggest_...` y **devuelve** el AUC de validación cruzada.
2. Un `study` que la maximiza.
3. `study.optimize(objetivo, n_trials=...)`.

Para que corra en clase, cada intento usa 3 folds y hacemos 20 intentos (1–2 minutos en Colab).

### ✏️ Completa — la función objetivo
Complete la línea de `learning_rate`: un número real entre 0.01 y 0.3, en **escala logarítmica** (`log=True`), igual que la de `l2_regularization`.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def objetivo(trial):
    params = {
        "learning_rate": ...,   # TODO: trial.suggest_float("learning_rate", 0.01, 0.3, log=True)
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 4, 63),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 200, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-3, 10, log=True),
    }
    modelo = HistGradientBoostingClassifier(**params, random_state=42)
    return cross_val_score(modelo, X_train, y_train, cv=cv3, scoring="roc_auc", n_jobs=-1).mean()

t = time.time()
estudio = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
estudio.optimize(objetivo, n_trials=20)
print(f"mejor AUC (3 folds): {estudio.best_value:.4f}   ({time.time() - t:.0f} s)")
estudio.best_params

In [ ]:
fg.optuna_historia([t.value for t in estudio.trials], optuna.importance.get_param_importances(estudio));

La línea roja sube rápido en los primeros intentos y después casi no se mueve. El panel derecho dice qué perillas importaron. Fíjese en la escala del eje: toda la búsqueda se mueve en un rango de **centésimas** de AUC.

## 4. 💥 El mejor de muchos sorteos

`best_score_` (o `estudio.best_value`) es **el máximo** entre muchas configuraciones, cada una medida con ruido. El máximo de muchas mediciones ruidosas siempre sale optimista. Con 36 000 filas el ruido es pequeño; con un dataset del tamaño de su proyecto, no.

Hagamos lo que haría cualquiera en su proyecto: 2 000 clientes, 60 configuraciones al azar, quedarse con la mejor. Y después, lo que casi nadie hace: medir **todas** en la prueba.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint

pocos = bank.sample(2000, random_state=1).index
Xp_train, Xp_test, yp_train, yp_test = train_test_split(X.loc[pocos], y.loc[pocos], test_size=0.3, random_state=42, stratify=y.loc[pocos])
distribuciones = {"learning_rate": loguniform(0.01, 0.3), "max_leaf_nodes": randint(4, 64), "min_samples_leaf": randint(5, 200),
                  "l2_regularization": loguniform(1e-3, 10), "max_iter": randint(50, 400)}
azar = RandomizedSearchCV(HistGradientBoostingClassifier(random_state=42), distribuciones, n_iter=60, cv=cv,
                          scoring="roc_auc", random_state=0, n_jobs=-1).fit(Xp_train, yp_train)
print(f"la ganadora de 60 configuraciones: AUC {azar.best_score_:.3f} en validación cruzada")

### 🔍 Predice
La ganadora prometió **0,770**. ¿Cuánto dará en la prueba? ¿Será también la mejor de las 60 en la prueba?

**Tu predicción:** AUC en prueba ≈ ____ · puesto ____ de 60

In [ ]:
resultados = pd.DataFrame(azar.cv_results_)
resultados["prueba"] = [roc_auc_score(yp_test, HistGradientBoostingClassifier(**p, random_state=42).fit(Xp_train, yp_train).predict_proba(Xp_test)[:, 1])
                        for p in resultados["params"]]
por_defecto = HistGradientBoostingClassifier(random_state=42)
defecto = (cross_val_score(por_defecto, Xp_train, yp_train, cv=cv, scoring="roc_auc").mean(),
           roc_auc_score(yp_test, por_defecto.fit(Xp_train, yp_train).predict_proba(Xp_test)[:, 1]))
g = azar.best_index_
print(f"ganadora: validación {resultados['mean_test_score'][g]:.3f} → prueba {resultados['prueba'][g]:.3f} "
      f"(puesto {(resultados['prueba'] > resultados['prueba'][g]).sum() + 1} de {len(resultados)} en la prueba)")
print(f"por defecto: validación {defecto[0]:.3f} → prueba {defecto[1]:.3f}")
fg.ganador_busqueda(resultados["mean_test_score"], resultados["prueba"], g, defecto=defecto);

## 5. La comparación honesta (con todos los datos)

De la línea base al boosting afinado con Optuna: validación cruzada (5 folds) y **una** evaluación en prueba de cada uno.

### ✏️ Completa — el modelo afinado
Cree `afinado` como un `HistGradientBoostingClassifier` con los mejores hiperparámetros de Optuna (`**estudio.best_params`) y `random_state=42`.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

afinado = ...   # TODO: HistGradientBoostingClassifier con **estudio.best_params y random_state=42
candidatos = {
    "línea base": DummyClassifier(strategy="prior"),
    "logística (S5)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "bosque (S6)": RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1),
    "boosting\npor defecto": HistGradientBoostingClassifier(random_state=42),
    "boosting\nafinado (Optuna)": afinado,
}
filas = {}
for nombre, m in candidatos.items():
    s = cross_val_score(m, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    filas[nombre] = (s.mean(), s.std(), roc_auc_score(y_test, m.fit(X_train, y_train).predict_proba(X_test)[:, 1]))
fg.comparacion_final(filas)
pd.DataFrame(filas, index=["CV promedio", "CV desv.", "prueba"]).T.round(3)

## 6. Una fila entra, una predicción sale: SHAP

Un boosting de cientos de árboles no se puede dibujar. **SHAP** (*SHapley Additive exPlanations*) reparte cada predicción entre las variables: empieza en el promedio de todos los clientes y suma lo que aporta cada variable para llegar a la predicción de **este** cliente. Es la "una fila" de la logística (Sesión 5), para cualquier modelo.

In [ ]:
import shap

explicador = shap.TreeExplainer(afinado)
muestra = X_test.iloc[:500]
valores = explicador(muestra)
base = float(np.ravel(explicador.expected_value)[0])
i = int(np.where(y_test.values[:500] == 1)[0][0])          # un cliente de prueba que sí abrió el depósito
fg.shap_una_fila(valores.values[i], base, muestra.iloc[i], X.columns, real=1);

In [ ]:
fg.shap_global(valores.values, X.columns);

**`contact_unknown`** (no se sabe por qué canal se le contactó) es la variable que más mueve las predicciones, seguida de **`poutcome_success`** (dijo sí en la campaña anterior). La primera es una pista para el banco: los registros incompletos coinciden con campañas menos efectivas. Recuerde la advertencia de siempre: SHAP dice qué **usa** el modelo, no qué **causa** que un cliente compre.

## 7. Guardar el modelo y usarlo con datos nuevos

En un proyecto real el modelo se usa después, con clientes nuevos que llegan **tal como vienen** (texto, sin *one-hot*). Por eso el modelo final se guarda como un `Pipeline` que incluye el preprocesamiento:

In [ ]:
import joblib
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder

crudas = bank.drop(columns=["deposit", "duration"])
categoricas = crudas.select_dtypes("object").columns.tolist()
modelo_final = make_pipeline(
    make_column_transformer((OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas), remainder="passthrough"),
    HistGradientBoostingClassifier(**estudio.best_params, random_state=42),
)
modelo_final.fit(crudas.loc[X_train.index], y_train)
joblib.dump(modelo_final, "modelo_bank.joblib")
print("guardado:", Path("modelo_bank.joblib").stat().st_size // 1024, "KB")

In [ ]:
cargado = joblib.load("modelo_bank.joblib")
cliente_nuevo = pd.DataFrame([{"age": 35, "job": "management", "marital": "single", "education": "tertiary", "default": "no",
                               "balance": 2500, "housing": "no", "loan": "no", "contact": "cellular", "day": 15, "month": "oct",
                               "campaign": 1, "pdays": 90, "previous": 2, "poutcome": "success"}])
print(f"probabilidad de que abra el depósito: {cargado.predict_proba(cliente_nuevo)[0, 1]:.0%}")

Un archivo, una línea para cargarlo, y el modelo recibe al cliente con sus columnas originales. Eso es lo que se entrega a quien lo va a usar (una aplicación, un tablero, otro equipo).

## 8. Demo opcional: AutoML con PyCaret 4

**PyCaret** automatiza en pocas líneas casi todo lo que hicimos en el curso: preprocesar, comparar modelos con validación cruzada, afinar. La versión 4 (alfa, 2026) funciona con el Colab actual; la 3 ya no.

Está **apagada** por defecto (`DEMO_PYCARET = False`) porque instalarla y correrla toma 2–3 minutos. Cámbiela a `True` si quiere verla.

In [ ]:
DEMO_PYCARET = False

if DEMO_PYCARET:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--pre", "pycaret==4.0.0a8"], check=True)
    from pycaret.classification import ClassificationExperiment
    experimento = ClassificationExperiment(target="deposit", session_id=42, train_size=0.8, fold=5).fit(bank.drop(columns=["duration"]))
    experimento.compare_models(include=["lr", "knn", "dt", "rf", "lightgbm"], sort="AUC")
    display(experimento.pull())
else:
    print("Demo de PyCaret apagada (DEMO_PYCARET = False).")

**Lo que muestra (corrida del profesor):**

| Modelo | AUC (5 folds) |
|---|---|
| LightGBM (boosting) | 0,792 |
| bosque | 0,779 |
| logística | **0,709** |
| árbol | 0,607 |
| k-NN | **0,611** |

**Lo que esconde:** la logística y k-NN salen mucho peor que en la Sesión 10 (0,763 y 0,742). ¿Por qué? Por defecto PyCaret **no escala** las variables (`normalize=False`) y codifica las categorías como números ordenados. Ya sabemos qué le pasa a k-NN sin escalar (Sesiones 5 y 7). La herramienta es rápida, pero sus valores por defecto toman decisiones que usted tiene que conocer: **AutoML no reemplaza saber lo que hay dentro del `.fit()`**.

## 9. 📦 Entrega E4 — el modelo final de su proyecto

Sobre **su** dataset (el de E1–E3), en un notebook de Colab:

| # | Sección | Qué debe contener |
|---|---|---|
| 1 | Pregunta y métrica | Recordatorio de E2: *y*, métrica elegida y por qué |
| 2 | Prueba apartada | `train_test_split` al inicio; no se toca hasta el punto 6 |
| 3 | `Pipeline` | Todo el preprocesamiento dentro (escalar, codificar, imputar, balancear) |
| 4 | Comparación con validación cruzada | Línea base + al menos 3 modelos (incluido uno de boosting), promedio ± desviación |
| 5 | Búsqueda de hiperparámetros | Grilla, azar u Optuna sobre el mejor modelo; qué perillas y rangos, y por qué |
| 6 | **Una** evaluación en prueba | Del modelo final; compárela con su validación cruzada y con E2 |
| 7 | Explicación | SHAP de una fila y global (o importancias), con la advertencia de qué **no** dice |
| 8 | Lectura honesta | ¿Cuánto ganó afinar? ¿Cuánto le gana a la línea base? ¿Qué limita el desempeño? |
| 9 | Modelo guardado | `joblib.dump` del `Pipeline` final y una celda que lo carga y predice una fila nueva |

Rúbrica (1–4 cada uno): corrección técnica · ausencia de fuga · honestidad en el reporte (prueba vs `best_score_`) · claridad.

Plantilla mínima (cambie lo que está en MAYÚSCULAS):

In [ ]:
# ---------- PLANTILLA E4 ----------
# df = pd.read_csv("/content/drive/MyDrive/SU_CARPETA/SUS_DATOS.csv")
# y = df["SU_Y"];  X = df.drop(columns=["SU_Y", "COLUMNAS_DEL_FUTURO", "IDENTIFICADORES"])
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# categoricas = X.select_dtypes("object").columns.tolist()
# prep = make_column_transformer((OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas), remainder="passthrough")
#
# def objetivo(trial):
#     m = make_pipeline(prep, HistGradientBoostingClassifier(learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
#                                                            max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 4, 63), random_state=42))
#     return cross_val_score(m, X_train, y_train, cv=5, scoring="SU_METRICA").mean()
# estudio = optuna.create_study(direction="maximize"); estudio.optimize(objetivo, n_trials=30)
#
# final = make_pipeline(prep, HistGradientBoostingClassifier(**estudio.best_params, random_state=42)).fit(X_train, y_train)
# print("prueba:", roc_auc_score(y_test, final.predict_proba(X_test)[:, 1]))   # ← ESTE es el número que se reporta
# joblib.dump(final, "mi_modelo.joblib")

## 10. Cierre del curso técnico

**Dentro del `.fit()` — los modelos del curso:**

| | Árbol | Lineal | Logística | k-NN | Bosque | **Boosting** | K-means | DBSCAN | PCA | t-SNE |
|---|---|---|---|---|---|---|---|---|---|---|
| ¿Usa *y*? | sí | sí | sí | sí | sí | sí | no | no | no | no |
| Qué minimiza | Gini | residuales² | log-loss | — | Gini por árbol | **log-loss (suma de árboles)** | inercia | — | − varianza | KL |
| Cómo lo busca | voraz | cerrada o gradiente | iterativo | guarda y vota | árboles en paralelo | **árboles en fila (gradiente)** | Lloyd | encadena | álgebra lineal | gradiente |
| Perilla clave | `max_depth` | `alpha` | `C` | `k` | `n_estimators` | **`learning_rate`, `max_leaf_nodes`** | `k` | `eps` | `n_components` | `perplexity` |

**Lo que se lleva del Módulo 4:** un número sin su incertidumbre no dice nada (S10); el preprocesamiento va dentro del `Pipeline` (S10); el mejor de muchos intentos es optimista y el número que se reporta es el de prueba (S11); casi toda la ganancia viene de buenos datos y un buen tipo de modelo, no de afinar perillas (S11).

**Sesión 12:** presentaciones del proyecto final (entre 3,5 y 5 minutos cada uno, según cuántos seamos; cuatro láminas): la pregunta, los datos y el pipeline, el resultado honesto y **qué aprendió de un error**. Traiga su modelo guardado con `joblib`: haremos una prueba de humo.